In [ ]:
!pip install transformers sentence-transformers torch scikit-learn Sastrawi nltk protobuf sentencepiece pandas

import os
import json
import re
import numpy as np
import torch
import logging
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Optional
from pathlib import Path
import nltk
nltk.download('punkt')
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering
from transformers import AutoTokenizer,AutoModel, AutoModelForSeq2SeqLM, MBartTokenizer, BertTokenizer
from sentence_transformers import SentenceTransformer
from google.colab import drive

logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.5 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
class TextPreprocessor:
    def __init__(self):
        factory = StemmerFactory()
        self.stemmer = factory.create_stemmer()

    def clean_text_display(self, text: str) -> str:
        # PEMBERSIHAN KHUSUS TAMPILAN

        remove = [r'^liputan6\.com\s*(?:-|:)\s*',r'^(?:liputan6\.com|com)\s*,\s*[a-z\s]+\s*(?:-|:)\s*',r'BACA JUGA:.*?(?=\n|$)', r'Disclaimer:.*?(?=\n|$)', r'^\s*ADVERTISEMENT.*?(?=\n|$)',
                  r'Perbesar', r'Selengkapnya', r'\d+\s+dari\s+\d+\s+halaman', r'\s*\([A-Za-z0-9/\.\s-]+\)\.?\s*$']
        for p in remove: text = re.sub(p, ' ', text, flags=re.IGNORECASE|re.DOTALL)

        # Fix Spasi Gancet
        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
        text = re.sub(r'([A-Z])([A-Z][a-z])', r'\1 \2', text)
        text = re.sub(r'([.,!?])(?=[a-zA-Z])', r'\1 ', text)
        text = re.sub(r'([a-zA-Z])(\d)', r'\1 \2', text)
        text = re.sub(r'(\d)([a-zA-Z])', r'\1 \2', text)
        text = text.encode('ascii', 'ignore').decode('ascii')

        return re.sub(r'\s+', ' ', text).strip()

    def case_folding_and_remove_artifacts(self, text: str) -> str:
        # TAHAP 3 & 4: CASE FOLDING & CLEANING KOMPUTASI
        text = text.lower()
        text = re.sub(r'http\S+|www\.\S+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        return re.sub(r'\s+', ' ', text).strip()

    def split_sentences(self, text: str) -> List[str]:
        try: return sent_tokenize(text)
        except: return re.split(r'(?<=[.!?])\s+', text)

In [ ]:
class SentenceBERTEmbedder:
    """
    Sentence-BERT (SBERT) paraphrase-multilingual-MiniLM-L12-v2 (Support Bahasa Indonesia)
    """
    def __init__(self, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        print(f"   [Model] Loading Sentence-BERT ({model_name})...")
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        try:
            self.model = SentenceTransformer(model_name, device=str(self.device))
        except Exception as e:
            print(f"  Gagal load SBERT: {e}")
            raise

    def get_embeddings(self, sentences: List[str]):
        if not sentences: return np.array([])
        embeddings = self.model.encode(sentences, convert_to_numpy=True, show_progress_bar=False)
        return embeddings

In [ ]:
# import numpy as np
# import pandas as pd
# from dataclasses import dataclass
# from typing import List, Optional

# @dataclass
# class Article:
#     url: str; title: str; content: str; category: str
#     date: Optional[str] = None; author: Optional[str] = None; tags: List[str] = None

# class MMRSelector:
#     def __init__(self, lambda_param=0.7):
#         self.lambda_param = lambda_param

#     def _cosine_similarity(self, A, B):
#         """
#         Rumus: (A . B) / (||A|| * ||B||)
#         """
#         dot_product = np.dot(A, B.T)
#         norm_a = np.linalg.norm(A, axis=1, keepdims=True)
#         norm_b = np.linalg.norm(B, axis=1, keepdims=True)
#         denominator = np.dot(norm_a, norm_b.T) + 1e-10
#         return dot_product / denominator

#     def select_with_logs(self, sentences_data, embs, query, n=5):
#         if not sentences_data: return [], [], []

#         embs = np.array(embs)
#         query = np.array(query)
#         if query.ndim == 1: query = query.reshape(1, -1)
#         if embs.ndim == 1: embs = embs.reshape(1, -1)

#         # 1. HITUNG RELEVANSI MURNI (SIM_1)
#         sim_to_query = self._cosine_similarity(embs, query).flatten()
#         raw_sim_to_query = sim_to_query.copy()

#         heuristic_logs = []

#         # 2. MODIFIKASI HEURISTIK (STATE TRACKER + 50% & 30% BOOST)
#         doc_quote_state = {}

#         for i, item in enumerate(sentences_data):
#             text = item['text'].lower() if isinstance(item, dict) else str(item).lower()

#             # Ambil identitas dokumen & urutan kalimatnya
#             doc_id = item.get('doc_id', 'unknown')
#             sent_id = item.get('sent_id', i)

#             if doc_id not in doc_quote_state:
#                 doc_quote_state[doc_id] = False

#             was_in_quote = doc_quote_state[doc_id]

#             straight_quotes = text.count('"')
#             has_curly_open = '“' in text
#             has_curly_close = '”' in text

#             rule_notes = []
#             multiplier = 1.0

#             # Quote Penalty Tracker
#             is_penalized = was_in_quote or (straight_quotes > 0) or has_curly_open or has_curly_close

#             if is_penalized:
#                 multiplier *= 0.70
#                 rule_notes.append("Quote Penalty (*0.70)")

#             # Update State Kutipan
#             if has_curly_open and not has_curly_close:
#                 doc_quote_state[doc_id] = True
#             elif has_curly_close and not has_curly_open:
#                 doc_quote_state[doc_id] = False
#             elif straight_quotes % 2 != 0:
#                 doc_quote_state[doc_id] = not doc_quote_state[doc_id]

#             # Position Boost 50% dan 30%
#             if sent_id == 1:
#                 multiplier *= 1.3
#                 rule_notes.append("Position Boost 2 (*1.30)")

#             sim_to_query[i] *= multiplier
#             applied_rule = " + ".join(rule_notes) if rule_notes else "Normal"

#             heuristic_logs.append({
#                 "Doc ID": doc_id,
#                 "Sent ID": sent_id,
#                 "Raw Sim_1": float(raw_sim_to_query[i]),
#                 "Multiplier": multiplier,
#                 "Modified Sim_1": float(sim_to_query[i]),
#                 "Rule Applied": applied_rule,
#                 "Text Snippet": text[:1000]
#             })

#         sel_idx = []
#         rem_idx = list(range(len(sentences_data)))
#         logs = []

#         # 3. LOOP MMR
#         while len(sel_idx) < n and rem_idx:
#             step_candidates = []

#             if sel_idx:
#                 selected_vecs = embs[sel_idx]

#             for i in rem_idx:
#                 rel = sim_to_query[i]
#                 div = 0.0
#                 if sel_idx:
#                     curr_vec = embs[i].reshape(1, -1)
#                     sim_to_selected = self._cosine_similarity(curr_vec, selected_vecs)
#                     div = np.max(sim_to_selected)

#                 score = self.lambda_param * rel - (1 - self.lambda_param) * div

#                 step_candidates.append({
#                     "idx": i, "score": score, "rel": rel, "div": div
#                 })

#             best = max(step_candidates, key=lambda x: x['score'])

#             sel_idx.append(best['idx'])
#             rem_idx.remove(best['idx'])
#             s = sentences_data[best['idx']]

#             doc_id = s.get('doc_id', '?')
#             sent_id = s.get('sent_id', '?')
#             source_str = f"[doc_{doc_id}, sent_{sent_id}]"

#             logs.append({
#                 "Step": len(sel_idx),
#                 "Source": source_str,
#                 "MMR Score": float(best['score']),
#                 "Modified Sim_1 (Rel)": float(best['rel']),
#                 "Diversity": float(best['div']),
#                 "Text": s['text'][:1000]
#             })

#         # sel_idx
#         selected_final = [sentences_data[i] for i in sel_idx]

#         return selected_final, logs, heuristic_logs

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Optional

class Article:
    url: str; title: str; content: str; category: str
    date: Optional[str] = None; author: Optional[str] = None; tags: List[str] = None

class MMRSelector:
    def __init__(self, lambda_param=0.7):
        self.lambda_param = lambda_param

    def _cosine_similarity(self, A, B):
        """
        Rumus: (A . B) / (||A|| * ||B||)
        """
        dot_product = np.dot(A, B.T)
        norm_a = np.linalg.norm(A, axis=1, keepdims=True)
        norm_b = np.linalg.norm(B, axis=1, keepdims=True)
        denominator = np.dot(norm_a, norm_b.T) + 1e-10
        return dot_product / denominator

    def select_with_logs(self, sentences_data, embs, query, n=5):
        if not sentences_data: return [], [], []

        embs = np.array(embs)
        query = np.array(query)
        if query.ndim == 1: query = query.reshape(1, -1)
        if embs.ndim == 1: embs = embs.reshape(1, -1)

        # 1. HITUNG RELEVANSI MURNI (SIM_1)
        sim_to_query = self._cosine_similarity(embs, query).flatten()

        heuristic_logs = []

        # 2. LOGGING BASELINE (Tanpa Modifikasi Heuristik)
        for i, item in enumerate(sentences_data):
            text = item['text'].lower() if isinstance(item, dict) else str(item).lower()

            # Ambil identitas dokumen & urutan kalimatnya
            doc_id = item.get('doc_id', 'unknown')
            sent_id = item.get('sent_id', i)

            # BASELINE: Multiplier selalu 1.0 (Skor murni SBERT)
            multiplier = 1.0

            heuristic_logs.append({
                "Doc ID": doc_id,
                "Sent ID": sent_id,
                "Raw Sim_1": float(sim_to_query[i]),
                "Multiplier": multiplier,
                "Modified Sim_1": float(sim_to_query[i]),
                "Rule Applied": "Baseline (Normal)",
                "Text Snippet": text[:1000]
            })

        sel_idx = []
        rem_idx = list(range(len(sentences_data)))
        logs = []

        # 3. LOOP MMR
        while len(sel_idx) < n and rem_idx:
            step_candidates = []

            if sel_idx:
                selected_vecs = embs[sel_idx]

            for i in rem_idx:
                rel = sim_to_query[i]
                div = 0.0
                if sel_idx:
                    curr_vec = embs[i].reshape(1, -1)
                    sim_to_selected = self._cosine_similarity(curr_vec, selected_vecs)
                    div = np.max(sim_to_selected)

                score = self.lambda_param * rel - (1 - self.lambda_param) * div

                step_candidates.append({
                    "idx": i, "score": score, "rel": rel, "div": div
                })

            best = max(step_candidates, key=lambda x: x['score'])

            sel_idx.append(best['idx'])
            rem_idx.remove(best['idx'])
            s = sentences_data[best['idx']]

            doc_id = s.get('doc_id', '?')
            sent_id = s.get('sent_id', '?')
            source_str = f"[doc_{doc_id}, sent_{sent_id}]"

            logs.append({
                "Step": len(sel_idx),
                "Source": source_str,
                "MMR Score": float(best['score']),
                "Modified Sim_1 (Rel)": float(best['rel']),
                "Diversity": float(best['div']),
                "Text": s['text'][:1000]
            })

        # sel_idx.sort()
        selected_final = [sentences_data[i] for i in sel_idx]

        return selected_final, logs, heuristic_logs

In [ ]:
print("⏳ Menginisialisasi Model...")
preprocessor = TextPreprocessor()
embedder = SentenceBERTEmbedder()
mmr = MMRSelector()
vectorizer = TfidfVectorizer(max_features=1000)
print(" Model Siap.\n")

⏳ Menginisialisasi Model...
   [Model] Loading Sentence-BERT (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Model Siap.



In [ ]:
import os
import shutil
from google.colab import drive

os.system('fusermount -uz /content/drive')

if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print("Warning: /content/drive still contains files after fusermount. Attempting to force clear.")
    try:
        shutil.rmtree('/content/drive')
        os.makedirs('/content/drive')
        print("/content/drive successfully removed and recreated.")
    except Exception as e:
        print(f"Error forcibly clearing /content/drive: {e}")

drive.mount('/content/drive', force_remount=True)

RAW_DATA_PATH = "/content/drive/MyDrive/Skripsi-Multi-Documents-Summary/data/raw/liputan6"
CATEGORY = "tekno" # custom kategori
CLUSTERING_THRESHOLD = 0.8

Mounted at /content/drive


In [ ]:
import pandas as pd

print("\nLoad judul artikel")
cat_dir = os.path.join(RAW_DATA_PATH, CATEGORY)
articles, raw_docs = [], []

if os.path.exists(cat_dir):
    for f in os.listdir(cat_dir):
        if f.endswith('.json'):
            try:
                d = json.load(open(os.path.join(cat_dir, f)))
                feat = f"{d['title']} {d['title']} {d.get('content','')[:500]}"
                articles.append(feat)
                raw_docs.append(d)
            except Exception as e:
                print(f"Skipping error file {f}: {e}")

if not articles: print("Data kosong"); exit()


Load judul artikel


In [ ]:
vecs = vectorizer.fit_transform(articles).toarray()

In [ ]:
# 2. CLUSTERING
clusterer = AgglomerativeClustering(
    n_clusters=None,
    metric='cosine',
    linkage='average',
    distance_threshold=CLUSTERING_THRESHOLD
)
labels = clusterer.fit_predict(vecs)

In [ ]:
# 3. GROUPING
clusters_dict = {}
for i, lbl in enumerate(labels):
    if lbl not in clusters_dict: clusters_dict[lbl] = []
    clusters_dict[lbl].append(raw_docs[i])

In [ ]:
#visualisasi cluster
print("PENGELOMPOKAN DOKUMEN (CLUSTERING)")
print("="*80)

clusters_list = []
valid_idx_counter = 0

sorted_lbls = sorted(clusters_dict.keys())
for lbl in sorted_lbls:
    docs = clusters_dict[lbl]
    is_valid = len(docs) >= 2

    if is_valid:
        status_icon = "VALID"
        clusters_list.append(docs)

        current_list_index = valid_idx_counter
        valid_idx_counter += 1

        index_msg = f"TARGET_INDEX = {current_list_index}"
    else:
        status_icon = "SKIP (Artikel < 2)"
        index_msg = "(Tidak masuk list analisis)"

    # Visualisasi Header
    status_icon = " VALID (Proses ke Summarization)" if is_valid else " SKIP (Artikel Tunggal/Kurang)"
    print(f"\n CLUSTER ID: {lbl}")
    print(f"   STATUS: {status_icon}")
    print(f"   JUMLAH ARTIKEL: {len(docs)}")
    print(f"   {index_msg}")
    print("   " + "-"*60)

    # List Judul
    for k, doc in enumerate(docs):
        print(f"   {k+1}. {doc['title']}")

    print("   " + "-"*60)

print("\n" + "="*85)
print(f" Total Klaster Valid tersimpan di 'clusters_list': {len(clusters_list)}")
print("="*85 + "\n")

PENGELOMPOKAN DOKUMEN (CLUSTERING)

 CLUSTER ID: 0
   STATUS:  VALID (Proses ke Summarization)
   JUMLAH ARTIKEL: 2
   TARGET_INDEX = 0
   ------------------------------------------------------------
   1. Harga PS5 Terbaru Januari 2026 di Indonesia, Varian Slim 1TB Turun Harga
   2. GTA 6 Meluncur November 2026, Xbox Terancam Kalah Pamor dari PS5
   ------------------------------------------------------------

 CLUSTER ID: 1
   STATUS:  VALID (Proses ke Summarization)
   JUMLAH ARTIKEL: 2
   TARGET_INDEX = 1
   ------------------------------------------------------------
   1. TV Portabel LG StanbyME 2 Bisa Dilepas Pasang, Ini Spesifikasi dan Harganya di Indonesia
   2. LG Indonesia akan Naikkan Harga TV hingga 7 Persen Imbas Krisis Chip Global
   ------------------------------------------------------------

 CLUSTER ID: 2
   STATUS:  VALID (Proses ke Summarization)
   JUMLAH ARTIKEL: 3
   TARGET_INDEX = 2
   ------------------------------------------------------------
   1. Menakar M

In [ ]:
# @title Konfigurasi & Load Target Klaster
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_colwidth', None)

# --- KONFIGURASI ---
USE_STEMMING = False
TARGET_CLUSTER_INDEX = 2


if not clusters_list:
    raise ValueError(" Error: Tidak ada klaster yang tersedia di 'clusters_list'!")

target_cluster = clusters_list[TARGET_CLUSTER_INDEX]

print(f"Klaster Topik Index [{TARGET_CLUSTER_INDEX}]")
print(f"Judul Utama: '{target_cluster[0]['title'][:500]}.'")
print(f"Jumlah Artikel dalam Klaster: {len(target_cluster)}")
print(f"Mode Stemming aktif: {USE_STEMMING}")

Klaster Topik Index [2]
Judul Utama: 'Menakar Masa Depan AI: Mengutamakan Dampak, Bukan Kompleksitas.'
Jumlah Artikel dalam Klaster: 3
Mode Stemming aktif: False


In [ ]:
#text cleaning & Sentence Splitting
print("[Processing] Memulai Preprocessing...\n")

preprocessor = TextPreprocessor()
all_data = []

for doc_idx, art in enumerate(target_cluster):
    original_text = art['content']

    clean_display = preprocessor.clean_text_display(original_text)

    sents = preprocessor.split_sentences(clean_display)

    print("="*60)
    print(f"🔍 DEMO TAHAPAN PREPROCESSING (DOKUMEN {doc_idx + 1})")
    print("="*60)
    print(f"🟢 1. TEKS MENTAH AWAL:\n{original_text[:1000000]}...\n")
    print(f"🟢 2. HASIL TAHAP 1 - DISPLAY CLEANSING:\n{clean_display[:1000000]}...\n")


    print(f"🟢 3. HASIL TAHAP 2 - SENTENCE SPLITTING (3 Kalimat Pertama):")
    for i, s in enumerate(sents[:3]):
        print(f"   [{i+1}] {s}")

    print(f"\n🟢 4. HASIL TAHAP 3 - LENGTH FILTERING & CASE FOLDING:")

    for sent_idx, s in enumerate(sents):
        s = s.strip()

        if len(s) >= 20 and re.search(r'[a-zA-Z]', s):

            calc_text = preprocessor.case_folding_and_remove_artifacts(s)

            if sent_idx < 3:
                print(f"   [Valid] -> {calc_text}")

            # Simpan ke memori
            meta = {
                'doc_id': doc_idx + 1,
                'sent_id': sent_idx + 1,
                'original_text': s,      # Teks ini untuk ditampilkan di ringkasan akhir
                'calc_text': calc_text   # Teks ini yang akan dikonversi SBERT
            }
            all_data.append(meta)

        else:
            if sent_idx < 3:
                print(f"   [Dibuang (Noise/Terlalu Pendek)] -> {s}")

    print("="*60, "\n")

print(f"✅ Preprocessing Selesai untuk seluruh dokumen.")
print(f"✅ Total Kalimat Valid: {len(all_data)} kalimat\n")

# Menampilkan hasil akhir DataFrame
print("📊 Sample Data Tersimpan (Format Input SBERT):")
df_prep = pd.DataFrame(all_data)[['doc_id', 'sent_id', 'original_text', 'calc_text']]
df_prep.columns = ['Doc ID', 'Sent ID', 'Kalimat Asli (Tampilan)', 'Kalimat Bersih (Input SBERT)']

# Menampilkan 10 baris pertama di tabel. Ubah head(10) menjadi display(df_prep) jika ingin melihat semua baris.
display(df_prep.head(10))

[Processing] Memulai Preprocessing...

🔍 DEMO TAHAPAN PREPROCESSING (DOKUMEN 1)
🟢 1. TEKS MENTAH AWAL:
Liputan6.com, Jakarta -Di tengah perlombaan global mengadopsi kecerdasan buatan (AI), talenta Indonesia mulai mengambil peran krusial di pusat inovasi dunia. Product Manager di Google Amerika Serikat, Juan Anugraha Djuwadi, kini menjadi salah satu aktor strategis yang mengarahkan bagaimana teknologi AI dikembangkan agar tetap manusiawi dan berdampak luas bagi pengguna global. Dalam webinar bertajuk “AI Streamline Your Business: Build Internal Apps with AI” baru-baru ini, Juan membedah peta jalan pengembangan AI yang melampaui sekadar kecanggihan teknis. Baginya, kunci utama efektivitas AI terletak pada kegunaannya dalam menjawab persoalan nyata di masyarakat. Juan, yang merupakan alumnus Columbia University, menegaskan bahwa pengguna akhir seringkali tidak mempedulikan kerumitan algoritma di balik sebuah aplikasi. “Pengguna tidak peduli seberapa canggih teknologi di belakang layar. Me

,Doc ID,Sent ID,Kalimat Asli (Tampilan),Kalimat Bersih (Input SBERT)
0,1,1,"Di tengah perlombaan global mengadopsi kecerdasan buatan (AI), talenta Indonesia mulai mengambil peran krusial di pusat inovasi dunia.",di tengah perlombaan global mengadopsi kecerdasan buatan ai talenta indonesia mulai mengambil peran krusial di pusat inovasi dunia
1,1,2,"Product Manager di Google Amerika Serikat, Juan Anugraha Djuwadi, kini menjadi salah satu aktor strategis yang mengarahkan bagaimana teknologi AI dikembangkan agar tetap manusiawi dan berdampak luas bagi pengguna global.",product manager di google amerika serikat juan anugraha djuwadi kini menjadi salah satu aktor strategis yang mengarahkan bagaimana teknologi ai dikembangkan agar tetap manusiawi dan berdampak luas bagi pengguna global
2,1,3,"Dalam webinar bertajuk AI Streamline Your Business: Build Internal Apps with AI baru-baru ini, Juan membedah peta jalan pengembangan AI yang melampaui sekadar kecanggihan teknis.",dalam webinar bertajuk ai streamline your business build internal apps with ai barubaru ini juan membedah peta jalan pengembangan ai yang melampaui sekadar kecanggihan teknis
3,1,4,"Baginya, kunci utama efektivitas AI terletak pada kegunaannya dalam menjawab persoalan nyata di masyarakat.",baginya kunci utama efektivitas ai terletak pada kegunaannya dalam menjawab persoalan nyata di masyarakat
4,1,5,"Juan, yang merupakan alumnus Columbia University, menegaskan bahwa pengguna akhir seringkali tidak mempedulikan kerumitan algoritma di balik sebuah aplikasi.",juan yang merupakan alumnus columbia university menegaskan bahwa pengguna akhir seringkali tidak mempedulikan kerumitan algoritma di balik sebuah aplikasi
5,1,6,Pengguna tidak peduli seberapa canggih teknologi di belakang layar.,pengguna tidak peduli seberapa canggih teknologi di belakang layar
6,1,7,"Mereka peduli apakah solusi itu berguna dan menyelesaikan masalah nyata, ujar Juan, dikutip Kamis (15/1/2026).",mereka peduli apakah solusi itu berguna dan menyelesaikan masalah nyata ujar juan dikutip kamis
7,1,8,"Prinsip ini diterjemahkan ke dalam dua filosofi kunci:less is moredanthe details matter.Menurutnya, dalam skala pengguna miliaran seperti di Google, detail sekecil apa pun menjadi isu strategis.",prinsip ini diterjemahkan ke dalam dua filosofi kunciless is moredanthe details mattermenurutnya dalam skala pengguna miliaran seperti di google detail sekecil apa pun menjadi isu strategis
8,1,9,Kegagalan sistem sebesar satu persen saja dapat berdampak pada jutaan orang.,kegagalan sistem sebesar satu persen saja dapat berdampak pada jutaan orang
9,1,10,Persfektif ini dinilai sangat relevan bagi pemerintah Indonesia dan perusahaan BUMN yang tengah mengintegrasikan AI dalam layanan publik berskala nasional.,persfektif ini dinilai sangat relevan bagi pemerintah indonesia dan perusahaan bumn yang tengah mengintegrasikan ai dalam layanan publik berskala nasional


In [ ]:
#text celaning & Sentence Splitting
print("[Processing]...")

all_data = []
calc_texts = []   # (clean/stemmed teks)

for doc_idx, art in enumerate(target_cluster):

    clean = preprocessor.clean_text_display(art['content'])

    sents = preprocessor.split_sentences(clean)

    for sent_idx, s in enumerate(sents):
        s = s.strip()
        if len(s) >= 20 and re.search(r'[a-zA-Z]', s):
            clean_artifact_text = preprocessor.case_folding_and_remove_artifacts(s)
            if USE_STEMMING:
                processed = preprocessor.stemming_process(clean_artifact_text)
            else:
                processed = clean_artifact_text

            meta = {
                'text': s,
                'doc_id': doc_idx + 1,
                'sent_id': sent_idx + 1,
                'calc_text': processed
            }
            all_data.append(meta)
            calc_texts.append(processed)

print(f" Preprocessing Selesai.")
print(f" Total Kalimat Valid: {len(all_data)} kalimat")

print("\n Sample Data (Original vs Calculation Input):")
df_prep = pd.DataFrame(all_data)[['text', 'calc_text']]
df_prep.columns = ['Original Text (Display)', 'Cleaned text for Embedding Input (SBERT)']
display(df_prep)

[Processing]...
 Preprocessing Selesai.
 Total Kalimat Valid: 115 kalimat

 Sample Data (Original vs Calculation Input):


,Original Text (Display),Cleaned text for Embedding Input (SBERT)
0,"Di tengah perlombaan global mengadopsi kecerdasan buatan (AI), talenta Indonesia mulai mengambil peran krusial di pusat inovasi dunia.",di tengah perlombaan global mengadopsi kecerdasan buatan ai talenta indonesia mulai mengambil peran krusial di pusat inovasi dunia
1,"Product Manager di Google Amerika Serikat, Juan Anugraha Djuwadi, kini menjadi salah satu aktor strategis yang mengarahkan bagaimana teknologi AI dikembangkan agar tetap manusiawi dan berdampak luas bagi pengguna global.",product manager di google amerika serikat juan anugraha djuwadi kini menjadi salah satu aktor strategis yang mengarahkan bagaimana teknologi ai dikembangkan agar tetap manusiawi dan berdampak luas bagi pengguna global
2,"Dalam webinar bertajuk AI Streamline Your Business: Build Internal Apps with AI baru-baru ini, Juan membedah peta jalan pengembangan AI yang melampaui sekadar kecanggihan teknis.",dalam webinar bertajuk ai streamline your business build internal apps with ai barubaru ini juan membedah peta jalan pengembangan ai yang melampaui sekadar kecanggihan teknis
3,"Baginya, kunci utama efektivitas AI terletak pada kegunaannya dalam menjawab persoalan nyata di masyarakat.",baginya kunci utama efektivitas ai terletak pada kegunaannya dalam menjawab persoalan nyata di masyarakat
4,"Juan, yang merupakan alumnus Columbia University, menegaskan bahwa pengguna akhir seringkali tidak mempedulikan kerumitan algoritma di balik sebuah aplikasi.",juan yang merupakan alumnus columbia university menegaskan bahwa pengguna akhir seringkali tidak mempedulikan kerumitan algoritma di balik sebuah aplikasi
...,...,...
110,"Potret HP Infinix Smart 10 Plus, HP Rp 1 jutaan terbaru yang hadir dengan dukungan AI hingga sertifikat anti debu dan air, IP 64 (Foto: Infinix).",potret hp infinix smart plus hp rp jutaan terbaru yang hadir dengan dukungan ai hingga sertifikat anti debu dan air ip foto infinix
111,"Infinix Smart 10 Plus hadir dengan pengalaman kecerdasan buatan (AI) yang menonjol, salah satu daya tariknya terletak pada kehadiran Folax, asisten virtual berbasis AI.",infinix smart plus hadir dengan pengalaman kecerdasan buatan ai yang menonjol salah satu daya tariknya terletak pada kehadiran folax asisten virtual berbasis ai
112,"Folax dirancang untuk mempermudah aktivitas harian pengguna, mampu menyusun pesan, menjadwalkan aktivitas, hingga mencari informasi dengan cepat.",folax dirancang untuk mempermudah aktivitas harian pengguna mampu menyusun pesan menjadwalkan aktivitas hingga mencari informasi dengan cepat
113,Ponsel ini dibekali baterai berkapasitas 6.000 m Ah dengan dukungan fast charging 18 W.,ponsel ini dibekali baterai berkapasitas m ah dengan dukungan fast charging w


In [ ]:
#Embedding Generation (Sentence-BERT)
import pandas as pd
import numpy as np

sent_vecs = embedder.get_embeddings(calc_texts)

query_vec = np.mean(sent_vecs, axis=0)


print(" INFORMASI STRUKTUR DATA:")
print(f"   • Tipe Data Output : {type(sent_vecs)}")
print(f"   • Matriks Kalimat  : {sent_vecs.shape}  -> (Total Kalimat, Dimensi Model)")
print(f"   • Vektor Query     : {query_vec.shape}     -> (Dimensi Model,)")

 INFORMASI STRUKTUR DATA:
   • Tipe Data Output : <class 'numpy.ndarray'>
   • Matriks Kalimat  : (115, 384)  -> (Total Kalimat, Dimensi Model)
   • Vektor Query     : (384,)     -> (Dimensi Model,)


In [ ]:
# (dimensi Sample 5 Kalimat Pertama)

num_preview_dims = 384
cols = [f"Dim_{i+1}" for i in range(num_preview_dims)]

if sent_vecs.ndim == 1:
    df_vec_preview = pd.DataFrame(sent_vecs[:num_preview_dims].reshape(1, -1), columns=cols)
    df_vec_preview.insert(0, "Text Snippet", [calc_text[:10000] ])
elif sent_vecs.ndim == 2:
    df_vec_preview = pd.DataFrame(sent_vecs[:100, :num_preview_dims], columns=cols)
    df_vec_preview.insert(0, "Text Snippet", [t[:1000] for t in calc_texts[:100]])
else:
    raise ValueError("sent_vecs has unexpected dimensions.")

display(df_vec_preview)

,Text Snippet,Dim_1,Dim_2,Dim_3,Dim_4,Dim_5,Dim_6,Dim_7,Dim_8,Dim_9,...,Dim_375,Dim_376,Dim_377,Dim_378,Dim_379,Dim_380,Dim_381,Dim_382,Dim_383,Dim_384
0,di tengah perlombaan global mengadopsi kecerdasan buatan ai talenta indonesia mulai mengambil peran krusial di pusat inovasi dunia,0.189700,0.081387,-0.249778,-0.070411,-0.109405,-0.168465,0.067418,0.196160,-0.086247,...,0.108329,-0.017193,0.018506,0.205614,-0.242774,0.133608,0.242542,0.170082,-0.124843,-0.024607
1,product manager di google amerika serikat juan anugraha djuwadi kini menjadi salah satu aktor strategis yang mengarahkan bagaimana teknologi ai dikembangkan agar tetap manusiawi dan berdampak luas bagi pengguna global,-0.217632,0.020319,-0.059410,-0.260153,-0.133210,-0.095813,0.198513,0.035066,-0.018904,...,0.189109,0.043128,-0.002502,-0.194602,-0.037253,0.265025,-0.016554,-0.284555,0.035442,0.128904
2,dalam webinar bertajuk ai streamline your business build internal apps with ai barubaru ini juan membedah peta jalan pengembangan ai yang melampaui sekadar kecanggihan teknis,-0.145410,-0.227322,-0.021296,-0.179999,0.148238,0.042999,-0.256261,-0.129809,-0.015015,...,0.122709,0.105238,-0.003897,-0.171966,0.039623,0.250100,-0.243303,0.139276,0.290651,0.088567
3,baginya kunci utama efektivitas ai terletak pada kegunaannya dalam menjawab persoalan nyata di masyarakat,-0.195242,-0.087658,-0.182792,0.014352,-0.080117,-0.143006,0.019054,0.245063,0.019270,...,0.100557,0.011179,0.196877,0.245222,-0.054827,0.111626,0.087833,0.112648,0.261143,0.152891
4,juan yang merupakan alumnus columbia university menegaskan bahwa pengguna akhir seringkali tidak mempedulikan kerumitan algoritma di balik sebuah aplikasi,-0.112157,-0.066843,-0.038625,-0.252461,-0.017068,-0.261504,-0.028892,0.275552,0.207704,...,0.301913,-0.067475,0.110268,-0.294967,-0.118493,0.054086,-0.115279,0.024385,0.311798,0.105448
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,infinix hot i infinix hot i hadir dengan serangkaian fitur kecerdasan buatan ai yang diklaim paling lengkap di kelasnya menjadikannya pilihan menarik di segmen harga terjangkau,-0.090537,0.027703,-0.069617,0.096545,-0.079783,0.006308,0.055922,0.030585,-0.107073,...,0.175018,-0.160670,0.126892,-0.002374,-0.088230,0.076905,0.411613,0.226063,-0.191593,0.193871
96,perangkat ini berjalan di atas sistem operasi infinix xos berbasis android yang dilengkapi dengan empat fitur ai yang disebut onetap infinix ai,-0.122954,-0.087596,-0.018776,-0.281992,-0.125379,0.082617,0.083261,-0.239590,0.133264,...,0.101358,0.135668,0.100846,-0.288046,0.108282,0.288790,-0.070975,0.185214,0.251210,0.205990
97,fiturfitur ini mencakup produktivitas sekali ketuk untuk membantu pengguna membuat daftar tugas catatan dan menjalankan perintah kerja ringan dengan cepat,-0.278670,-0.035580,-0.136952,-0.029451,-0.307695,-0.062887,0.091451,0.211475,0.000005,...,0.143827,0.143560,0.218103,-0.271882,0.293347,0.143143,0.013824,0.439754,-0.093892,0.041276
98,selain itu infinix hot i juga dibekali onetap summarize sebuah fitur yang sangat berguna untuk merangkum teks panjang menjadi poinpoin penting yang mudah dipahami dan onetap answers yang memberikan feedback atau respons instan terhadap pertanyaan umum atau perintah teks,-0.284479,-0.130289,0.060833,0.119596,0.043076,0.097192,0.039742,0.161428,0.062741,...,-0.079612,0.054321,0.208100,0.041677,0.368569,0.126966,0.052585,0.347566,-0.152549,0.043721


In [ ]:
# Query Vector
print("\n PREVIEW QUERY VECTOR (CENTROID) (10 Dimensi Awal):")
print("   (Rata-rata dari seluruh vektor di atas, merepresentasikan 'topik inti')")
df_query = pd.DataFrame(query_vec.reshape(1, -1)[:, :500], columns=[f"Dim_{i+1}" for i in range(384)])
display(df_query)

# Nilai
print("\n STATISTIK NILAI VEKTOR:")
print(f"   • Nilai Min dalam Matrix : {np.min(sent_vecs):.4f}")
print(f"   • Nilai Max dalam Matrix : {np.max(sent_vecs):.4f}")
print(f"   • Nilai Mean (Rata-rata) : {np.mean(sent_vecs):.4f}")


 PREVIEW QUERY VECTOR (CENTROID) (10 Dimensi Awal):
   (Rata-rata dari seluruh vektor di atas, merepresentasikan 'topik inti')


,Dim_1,Dim_2,Dim_3,Dim_4,Dim_5,Dim_6,Dim_7,Dim_8,Dim_9,Dim_10,...,Dim_375,Dim_376,Dim_377,Dim_378,Dim_379,Dim_380,Dim_381,Dim_382,Dim_383,Dim_384
0,-0.049534,0.067531,-0.109467,-0.124708,0.015067,-0.029547,-0.077547,0.019175,0.005123,0.052746,...,0.073823,0.004224,0.052632,-0.04323,-0.011829,0.109979,0.061422,0.033432,0.107196,0.108115



 STATISTIK NILAI VEKTOR:
   • Nilai Min dalam Matrix : -0.8169
   • Nilai Max dalam Matrix : 1.0014
   • Nilai Mean (Rata-rata) : 0.0002


In [ ]:
# MMR Selection Process (Ranking)

# COMPRESSION_RATE = 0.20
# COMPRESSION_RATE = 0.30
# COMPRESSION_RATE = 0.40
COMPRESSION_RATE = 0.50
# n_teratas= 8

total_sentences_input = len(all_data)
jumlah_terkompresi = int(total_sentences_input * COMPRESSION_RATE)

# selected_data, logs = mmr.select_with_logs(all_data, sent_vecs, query_vec, n=jumlah_terkompresi)
selected_data, logs, heuristic_logs = mmr.select_with_logs(all_data, sent_vecs, query_vec, n=jumlah_terkompresi)

print(f"✅ Jumlah Kalimat Terpilih: {len(selected_data)} dari total {total_sentences_input} kalimat.\n")

print("\n" + "="*80)
df_heuristics = pd.DataFrame(heuristic_logs)
display(df_heuristics)

print("\n" + "="*80)
print("\n TABEL DETAIL SCORING MMR :")
df_logs = pd.DataFrame(logs)

#Output
# df_logs = df_logs[['Step', 'Source', 'MMR Score', 'Relevance', 'Diversity', 'Text']]
df_logs = pd.DataFrame(logs)
display(df_logs)

✅ Jumlah Kalimat Terpilih: 57 dari total 115 kalimat.




,Doc ID,Sent ID,Raw Sim_1,Multiplier,Modified Sim_1,Rule Applied,Text Snippet
0,1,1,0.556876,1.0,0.556876,Baseline (Normal),"di tengah perlombaan global mengadopsi kecerdasan buatan (ai), talenta indonesia mulai mengambil peran krusial di pusat inovasi dunia."
1,1,2,0.603104,1.0,0.603104,Baseline (Normal),"product manager di google amerika serikat, juan anugraha djuwadi, kini menjadi salah satu aktor strategis yang mengarahkan bagaimana teknologi ai dikembangkan agar tetap manusiawi dan berdampak luas bagi pengguna global."
2,1,3,0.636507,1.0,0.636507,Baseline (Normal),"dalam webinar bertajuk ai streamline your business: build internal apps with ai baru-baru ini, juan membedah peta jalan pengembangan ai yang melampaui sekadar kecanggihan teknis."
3,1,4,0.617614,1.0,0.617614,Baseline (Normal),"baginya, kunci utama efektivitas ai terletak pada kegunaannya dalam menjawab persoalan nyata di masyarakat."
4,1,5,0.516686,1.0,0.516686,Baseline (Normal),"juan, yang merupakan alumnus columbia university, menegaskan bahwa pengguna akhir seringkali tidak mempedulikan kerumitan algoritma di balik sebuah aplikasi."
...,...,...,...,...,...,...,...
110,3,35,0.490535,1.0,0.490535,Baseline (Normal),"potret hp infinix smart 10 plus, hp rp 1 jutaan terbaru yang hadir dengan dukungan ai hingga sertifikat anti debu dan air, ip 64 (foto: infinix)."
111,3,36,0.613007,1.0,0.613007,Baseline (Normal),"infinix smart 10 plus hadir dengan pengalaman kecerdasan buatan (ai) yang menonjol, salah satu daya tariknya terletak pada kehadiran folax, asisten virtual berbasis ai."
112,3,37,0.410591,1.0,0.410591,Baseline (Normal),"folax dirancang untuk mempermudah aktivitas harian pengguna, mampu menyusun pesan, menjadwalkan aktivitas, hingga mencari informasi dengan cepat."
113,3,38,0.263359,1.0,0.263359,Baseline (Normal),ponsel ini dibekali baterai berkapasitas 6.000 m ah dengan dukungan fast charging 18 w.




 TABEL DETAIL SCORING MMR :


,Step,Source,MMR Score,Modified Sim_1 (Rel),Diversity,Text
0,1,"[doc_2, sent_51]",0.505339,0.721913,0.000000,"Dalam skala yang lebih luas, Lenovo mengidentifikasi sejumlah faktor kunci keberhasilan penerapan AI."
1,2,"[doc_2, sent_5]",0.363250,0.700187,0.422936,Perusahaan mulai mencari cara agar teknologi ini benar-benar berdampak bagi bisnis.
2,3,"[doc_3, sent_1]",0.335905,0.705816,0.527220,Tren kecerdasan buatan (AI) semakin merambah ke segmensmartphone entry-leveldan menengah.
3,4,"[doc_2, sent_59]",0.328440,0.717482,0.579326,Perangkat-perangkat yang kini dibawa Lenovo ke pasar telah memiliki kapabilitas AI.
4,5,"[doc_1, sent_16]",0.309667,0.650407,0.485395,"Jika AS sudah mapan dalam hal monetisasi perangkat lunak, Indonesia masih berada dalam tahap transisi yang dinamis."
5,6,"[doc_2, sent_48]",0.307778,0.693046,0.591178,"Dari 2025 ke 2026, tahap implementasi dan uji coba AI meningkat menjadi 66 persen."
6,7,"[doc_1, sent_14]",0.296221,0.678077,0.594778,"Pendekatan ini menjadi krusial bagi para regulator dan pemimpin perusahaan di Asia Pasifik (APAC) agar tidak hanya bersikap reaktif terhadap tren, tetapi mampu menjadi visioner dalam merancang ekosistem digital."
7,8,"[doc_2, sent_6]",0.292543,0.629039,0.492613,"Bukan lagi soal membangun model AI, tetapi bagaimana mengintegrasikannya dengan perangkat, infrastruktur, dan sistem sudah berjalan."
8,9,"[doc_2, sent_58]",0.292297,0.660792,0.567525,"Sejalan dengan hal ini, prioritas investasi TI pun bergeser ke arah penerapan perangkay AI guna meningkatkan produktivitas dan mendukung proses inferensi AI secara lokal."
9,10,"[doc_2, sent_60]",0.291337,0.663600,0.577276,"Artinya, perangkat tersebut memiliki kemampuan dan kinerja yang diperlukan untuk menjalankan model AI secara lokal di sisi edge ."


In [ ]:

final_text_parts = []

for item in selected_data:
    final_text_parts.append(item['text'])

extractive_clean = " ".join(final_text_parts)

print("\n Hasil Ekstraktif:")
print("-" * 75)
print(extractive_clean)
print("-" * 75)

inp_model = " ".join(extractive_clean.split()[:512])
print(f"\n Input ke Model Abstractive Siap.")
print(f"   Jumlah Kata: {len(inp_model.split())}")


 Hasil Ekstraktif:
---------------------------------------------------------------------------
Dalam skala yang lebih luas, Lenovo mengidentifikasi sejumlah faktor kunci keberhasilan penerapan AI. Perusahaan mulai mencari cara agar teknologi ini benar-benar berdampak bagi bisnis. Tren kecerdasan buatan (AI) semakin merambah ke segmensmartphone entry-leveldan menengah. Perangkat-perangkat yang kini dibawa Lenovo ke pasar telah memiliki kapabilitas AI. Jika AS sudah mapan dalam hal monetisasi perangkat lunak, Indonesia masih berada dalam tahap transisi yang dinamis. Dari 2025 ke 2026, tahap implementasi dan uji coba AI meningkat menjadi 66 persen. Pendekatan ini menjadi krusial bagi para regulator dan pemimpin perusahaan di Asia Pasifik (APAC) agar tidak hanya bersikap reaktif terhadap tren, tetapi mampu menjadi visioner dalam merancang ekosistem digital. Bukan lagi soal membangun model AI, tetapi bagaimana mengintegrasikannya dengan perangkat, infrastruktur, dan sistem sudah berjalan. 

In [ ]:
!pip install rouge_score




In [ ]:


import pandas as pd
from rouge_score import rouge_scorer


ground_truth_text_1 = """
Di tengah perlombaan global mengadopsi kecerdasan buatan (AI), talenta Indonesia mulai mengambil peran krusial di pusat inovasi dunia. Product Manager di Google Amerika Serikat, Juan Anugraha Djuwadi, kini menjadi salah satu aktor strategis yang mengarahkan bagaimana teknologi AI dikembangkan agar tetap manusiawi dan berdampak luas bagi pengguna global. Dalam webinar bertajuk “AI Streamline Your Business: Build Internal Apps with AI” baru-baru ini, Juan membedah peta jalan pengembangan AI yang melampaui sekadar kecanggihan teknis. Baginya, kunci utama efektivitas AI terletak pada kegunaannya dalam menjawab persoalan nyata di masyarakat. Juan, yang merupakan alumnus Columbia University, menegaskan bahwa pengguna akhir seringkali tidak mempedulikan kerumitan algoritma di balik sebuah aplikasi. “Pengguna tidak peduli seberapa canggih teknologi di belakang layar. Mereka peduli apakah solusi itu berguna dan menyelesaikan masalah nyata,” ujar Juan, dikutip Kamis (15/1/2026). Prinsip ini diterjemahkan ke dalam dua filosofi kunci: “less is more” dan “the details matter.” Menurutnya, dalam skala pengguna miliaran seperti di Google, detail sekecil apa pun menjadi isu strategis. Kegagalan sistem sebesar satu persen saja dapat berdampak pada jutaan orang. Persfektif ini dinilai sangat relevan bagi pemerintah Indonesia dan perusahaan BUMN yang tengah mengintegrasikan AI dalam layanan publik berskala nasional. Terkait proses pengambilan keputusan, Juan menawarkan pendekatan moderat antara data dan intuisi. Ia mengibaratkan data sebagai kompas untuk optimasi efisiensi dan akurasi, namun terobosan besar tetap membutuhkan visi manusia. “Data memvalidasi masa kini, namun intuisi mendefinisikan masa depan,” ungkapnya. Pendekatan ini menjadi krusial bagi para regulator dan pemimpin perusahaan di Asia Pasifik (APAC) agar tidak hanya bersikap reaktif terhadap tren, tetapi mampu menjadi visioner dalam merancang ekosistem digital. Menyoroti lanskap digital di Tanah Air, Juan melihat adanya perbedaan fase kematangan antara pasar Amerika Serikat (AS) dan Indonesia. Jika AS sudah mapan dalam hal monetisasi perangkat lunak, Indonesia masih berada dalam tahap transisi yang dinamis. Ia memprediksi, seiring berkembangnya ekosistem digital di Indonesia, isu mengenai privasi data, etika AI, dan akuntabilitas akan menjadi agenda utama yang tak terelakkan. Inovasi AI yang sukses di wilayah dengan keberagaman sosial tinggi seperti Indonesia dan APAC haruslah bersifat kontekstual dan berbasis pada kepercayaan. Mantan profesional di Niantic dan Activision ini juga memproyeksikan pergeseran fundamental dalam lima tahun mendatang melalui demokratisasi AI. Ia memperkirakan lahirnya era “software on-the-fly”, di mana perangkat lunak dihasilkan secara real-time sesuai kebutuhan instan pengguna, baik melalui antarmuka teks maupun suara. Ia membuktikan bahwa peran talenta Indonesia bukan sekadar pelengkap, melainkan kontributor strategis yang turut menentukan arah teknologi masa depan yang beretika dan berorientasi pada kepentingan manusia.
""".strip()

ground_truth_text_2 = """
""".strip()

if 'selected_data' in globals() and selected_data:
    system_summary_text = " ".join([item['text'] for item in selected_data])
else:
    print("WARNING: Variable 'selected_data' kosong/tidak ditemukan.")

if system_summary_text and ground_truth_text_1:
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    scores = scorer.score(ground_truth_text_1, system_summary_text)

    print("\n" + "="*60)
    print("HASIL EVALUASI AKHIR (ROUGE SCORE)")
    print(f"Lambda = {mmr.lambda_param}")
    print("="*60)

    print(f"\n SYSTEM SUMMARY (MMR):\n{system_summary_text}")
    print("-" * 60)
    print(f" GROUND TRUTH (Dari Annotator):\n{ground_truth_text_1}")
    print("-" * 60)

    results_data = {
        "Metric": ["ROUGE-1 (Kata)", "ROUGE-2 (Frasa)", "ROUGE-L (Urutan)"],
        "Precision": [scores['rouge1'].precision, scores['rouge2'].precision, scores['rougeL'].precision],
        "Recall": [scores['rouge1'].recall, scores['rouge2'].recall, scores['rougeL'].recall],
        "F1-Score": [scores['rouge1'].fmeasure, scores['rouge2'].fmeasure, scores['rougeL'].fmeasure]
    }

    df_results = pd.DataFrame(results_data)

    print("\nTABEL SKOR:")
    display(df_results.style.format("{:.4f}", subset=["Precision", "Recall", "F1-Score"]))

    f1_r1 = scores['rouge1'].fmeasure
    f1_r2 = scores['rouge2'].fmeasure
    f1_rl = scores['rougeL'].fmeasure
else:
    print(" Evaluasi dibatalkan karena data kosong.")


HASIL EVALUASI AKHIR (ROUGE SCORE)
Lambda = 0.7

 SYSTEM SUMMARY (MMR):
Dalam skala yang lebih luas, Lenovo mengidentifikasi sejumlah faktor kunci keberhasilan penerapan AI. Perusahaan mulai mencari cara agar teknologi ini benar-benar berdampak bagi bisnis. Tren kecerdasan buatan (AI) semakin merambah ke segmensmartphone entry-leveldan menengah. Perangkat-perangkat yang kini dibawa Lenovo ke pasar telah memiliki kapabilitas AI. Jika AS sudah mapan dalam hal monetisasi perangkat lunak, Indonesia masih berada dalam tahap transisi yang dinamis. Dari 2025 ke 2026, tahap implementasi dan uji coba AI meningkat menjadi 66 persen. Pendekatan ini menjadi krusial bagi para regulator dan pemimpin perusahaan di Asia Pasifik (APAC) agar tidak hanya bersikap reaktif terhadap tren, tetapi mampu menjadi visioner dalam merancang ekosistem digital. Bukan lagi soal membangun model AI, tetapi bagaimana mengintegrasikannya dengan perangkat, infrastruktur, dan sistem sudah berjalan. Sejalan dengan hal ini,

,Metric,Precision,Recall,F1-Score
0,ROUGE-1 (Kata),0.3252,0.8058,0.4634
1,ROUGE-2 (Frasa),0.2461,0.6107,0.3508
2,ROUGE-L (Urutan),0.1195,0.2961,0.1703
